### **Required Packages**

In [1]:
#### ------------------
## Data Manipulation
#### ------------------
import pandas as pd

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

## In .ipynb files, you can use this:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
main_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
# print(f"Main path: {main_path}")

#### ------------------
## Machine learning
#### ------------------
from sklearn import model_selection, metrics
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import  LogisticRegression
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

### Functions

In [2]:
##---------------------------------------------------------------------------------
# Function that evaluates the algorithm and returns metrics for classification models
##---------------------------------------------------------------------------------

## cv = 10
## Defines the cross-validation strategy.
## Specifies the number of folds to be performed.

## n_jobs = -1
## Number of jobs to be executed in parallel.
## Model training and prediction are parallelized across the cross-validation folds.
## -1 means that all available processors will be used.

def evaluate_models(algorithm, X_train, y_train, cv_folds):
    model = algorithm.fit(X_train, y_train)
    
    # Predicted classes for Accuracy and F1-Score
    train_predictions = model_selection.cross_val_predict(
        algorithm,
        X_train,
        y_train,
        cv=cv_folds,
        n_jobs=-1,
        method='predict'
    )
    
    # Scores for ROC AUC
    if hasattr(algorithm, 'predict_proba'):
        train_scores = model_selection.cross_val_predict(
            algorithm,
            X_train,
            y_train,
            cv=cv_folds,
            n_jobs=-1,
            method='predict_proba'
        )[:, 1]
    else:
        train_scores = model_selection.cross_val_predict(
            algorithm,
            X_train,
            y_train,
            cv=cv_folds,
            n_jobs=-1,
            method='decision_function'
        )
    
    accuracy = round(metrics.accuracy_score(y_train, train_predictions), 4)
    f1_score = round(metrics.f1_score(y_train, train_predictions), 4)
    roc_auc = round(metrics.roc_auc_score(y_train, train_scores), 4)
    
    return accuracy, f1_score, roc_auc

### **Loading the training data**


In [3]:
# Relative path to the training dataset
training_data_path = os.path.join(main_path, 'data', 'titanic_train.csv')
train = pd.read_csv(training_data_path, index_col=0)
train.head()

,Survived,Age_treated,SibSp,Parch,Fare_treated,Sex,Pclass_2,Pclass_3,Embarked_Q,Embarked_S
0,0,22,1,0,7.2500,0,False,True,False,True
1,1,38,1,0,71.2833,1,False,False,False,False
2,1,26,0,0,7.9250,1,False,True,False,True
3,1,35,1,0,53.1000,1,False,False,False,True
4,0,35,0,0,8.0500,0,False,True,False,True


In [4]:
### **Separating the training X and y variables**
x_train = train.drop('Survived', axis = 1)
y_train = train['Survived']
x_train.head()

,Age_treated,SibSp,Parch,Fare_treated,Sex,Pclass_2,Pclass_3,Embarked_Q,Embarked_S
0,22,1,0,7.2500,0,False,True,False,True
1,38,1,0,71.2833,1,False,False,False,False
2,26,0,0,7.9250,1,False,True,False,True
3,35,1,0,53.1000,1,False,False,False,True
4,35,0,0,8.0500,0,False,True,False,True


### **Testing Multiple Models**


In [5]:
##--------------------------
## **Logistic Regression**
##--------------------------

algorithm = LogisticRegression(max_iter=1000)
cv_folds = 10

accuracy, f1_score, roc_auc = evaluate_models(
                    algorithm, x_train, y_train, cv_folds
    )

print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.8025
F1-Score: 0.7284
Area Under the ROC Curve: 0.8489


In [6]:
##--------------------------
## KNN
##--------------------------
algorithm = KNeighborsClassifier()
cv_folds = 10

accuracy, f1_score, roc_auc = evaluate_models(
    algorithm, x_train, y_train, cv_folds
)

print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.6936
F1-Score: 0.5741
Area Under the ROC Curve: 0.7337


In [7]:
##--------------------------
## SVM
##--------------------------
## dual=False
## Selects the algorithm to solve the dual or primal optimization problem.
## Prefer dual=False when n_samples > n_features.
## dual="auto" automatically determines the value of the parameter
## based on the values of n_samples, n_features, loss, multi_class, and penalty.
## If n_samples < n_features and the optimizer supports the selected loss,
## multi_class, and penalty, then dual will be set to True; otherwise,
## it will be set to False.

algorithm = LinearSVC(dual=False)
cv_folds = 10
accuracy, f1_score, roc_auc = evaluate_models(algorithm, x_train, y_train, cv_folds)
print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.8002
F1-Score: 0.727
Area Under the ROC Curve: 0.8485


In [8]:
##--------------------------
## Random Forest
##--------------------------
algorithm = RandomForestClassifier()
cv_folds = 10
accuracy, f1_score, roc_auc = evaluate_models(algorithm, x_train, y_train, cv_folds)
print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.8159
F1-Score: 0.7545
Area Under the ROC Curve: 0.8584


In [9]:
##--------------------------
## Gradient Boost Trees
##--------------------------
algorithm = GradientBoostingClassifier()
cv_folds = 10
accuracy, f1_score, roc_auc = evaluate_models(algorithm, x_train, y_train, cv_folds)
print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.826
F1-Score: 0.7551
Area Under the ROC Curve: 0.8635


In [10]:
##--------------------------
## XGBoost
##--------------------------
## objective -> Specify the learning task and the corresponding learning objective or a custom objective function to be used
## random_state -> Random number seed.
algorithm = xgb.XGBClassifier(objective="binary:logistic", random_state=123)
cv_folds = 10
accuracy, f1_score, roc_auc = evaluate_models(algorithm, x_train, y_train, cv_folds)
print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.8193
F1-Score: 0.7593
Area Under the ROC Curve: 0.8613


In [11]:
##--------------------------
## MLP - Redes Neurais
##--------------------------
algorithm = MLPClassifier(random_state=1,
                          hidden_layer_sizes = (1024,128),
                          solver = "adam", 
                          activation = "logistic",
                          max_iter=300).fit(x_train, y_train)
cv_folds = 10
accuracy, f1_score, roc_auc = evaluate_models(algorithm, x_train, y_train, cv_folds)
print('Accuracy:', accuracy)
print('F1-Score:', f1_score)
print('Area Under the ROC Curve:', roc_auc)

Accuracy: 0.7621
F1-Score: 0.6748
Area Under the ROC Curve: 0.8108


Among the classification models tested, **Gradient Boosting and XGBoost achieved the best overall performance**. Gradient Boosting obtained the highest Accuracy (0.8260) and ROC AUC (0.8635), while XGBoost achieved the highest F1-Score (0.7593). The differences between the two models were relatively small, indicating that both provided competitive predictive performance.

Although Gradient Boosting achieved a slightly higher ROC AUC, **XGBoost achieved the highest F1-Score** among all the models evaluated. This metric is particularly relevant for a binary classification problem because it provides a balance between precision and recall, allowing the model's ability to correctly identify the positive class while minimizing both false positives and false negatives to be assessed.

In addition to its competitive predictive performance, XGBoost offers several practical advantages over conventional Gradient Boosting implementations. It incorporates **regularization mechanisms** that can help control model complexity and reduce the risk of overfitting. It also provides greater flexibility through a wide range of hyperparameters, allowing the model to be fine-tuned according to the characteristics of the dataset. Furthermore, XGBoost includes several computational optimizations and supports parallel processing during parts of the training process, making it efficient and scalable.

Considering both its predictive performance and its flexibility for optimization, **XGBoost was selected as the model to be further developed and tuned**. The next step is therefore to perform hyperparameter tuning in order to identify the parameter configuration that provides the best performance and subsequently evaluate the optimized model on the test set.


In [12]:
##---------------------------------------------
## Some notes about tunning xgboost parameters
##---------------------------------------------

##------------------------------------------------------------------------------------
## max_depth:
## Shallow trees are expected to have poor performance because they capture few details 
## of the problem and are generally referred to as weak learners. Deeper trees generally 
## capture too many details of the problem and overfit the training dataset, limiting 
## the ability to make good predictions on new data.
##------------------------------------------------------------------------------------
## n_estimators:
## With boosted tree models, models are trained sequentially - where each subsequence 
## tree tries to correct for the errors made by the previous sequence of trees.
## The n_estimators_ parameter specifies how many sequential trees we want to make 
## that attempt to correct for prior trees.

##------------------------------------------------------------------------------------
## learning_rate:
## The learning_rate parameter (also referenced in XGboost documentation as eta) controls 
## the magnitude of change that is permitted from one tree to the next.


params = dict( 
    learning_rate = [0.001, 0.010, 0.100, 0.500], 
    max_depth = [3,5,7,9, 10],
    n_estimators = [n for n in range(50, 400, 50)],
)

In [13]:
###--------------------------
## Cross-Validation
###--------------------------
algorithm = xgb.XGBClassifier(objective="binary:logistic",
                              seed=123)

xgb_cv = GridSearchCV(
    estimator=algorithm,
    param_grid=params,
    cv=10,
    scoring='f1',
    n_jobs=-1
)

xgb_cv.fit(x_train, y_train)

print(f"Best Score: {xgb_cv.best_score_}")
print(f"Best Parameters: {xgb_cv.best_estimator_}")

Best Score: 0.7697518236402037
Best Parameters: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)


In [14]:
# Relative path to the testing dataset
testing_data_path = os.path.join(main_path, 'data', 'titanic_test.csv')
x_test = pd.read_csv(testing_data_path, index_col=0)

## Prediction with test data
y_pred = xgb_cv.predict(x_test)
print(y_pred)

[0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 1 1 1 0 1 0 1 0 0 0 1 0 1 0 0
 0 0 1 0 1 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 1 1 0 0 0
 1 0 0 1 0 1 1 0 0 0 0 0 1 1 1 1 0 0 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 1 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0
 1 0 1 0 0 1 0 0 1 1 1 1 1 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 1 0 1 1 0 1
 0 1 0 0 0 0 0 1 0 1 0 1 1 0 0 1 1 0 1 0 0 0 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0
 1 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1
 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 1 0 1 1 0 1 0 0 0 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 0 0
 0 1 1 1 1 0 0 1 0 0 1]


The objective of this project is not to participate in the Kaggle competition, but rather to test different classification models as part of my personal learning process. However, I decided to include the final step of saving the DataFrame in the format required by Kaggle for competition submission.


In [15]:
## Reading the original test data to retrieve the `PassengerId` and concatenate it with the predictions.
# Relative path to the testing dataset
test_data_path = os.path.join(main_path, 'data', 'test.csv')

test = pd.read_csv(test_data_path)
# Saving PassengerId to assist in creating the file to be submitted to Kaggle.
passengerId = test['PassengerId']
print(passengerId)

0       892
1       893
2       894
3       895
4       896
       ... 
413    1305
414    1306
415    1307
416    1308
417    1309
Name: PassengerId, Length: 418, dtype: int64


In [16]:
kaggle = pd.DataFrame({'PassengerId': passengerId, 'Survived': y_pred})
kaggle.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [17]:
### Saving the dataset
# Relative path
kaggle_data_path = os.path.join(main_path, 'data', 'kaggle.csv')
kaggle.to_csv(kaggle_data_path, index=False)